In [1]:
import numpy as np
import pandas as pd
from recs import WeightedSimilarity

## Generate synthetic data

In [11]:
rng = np.random.default_rng(42)

N_CUSTOMERS = 50000
N_PRODUCTS = 200
N_INTERACTIONS = 150000

customers = pd.DataFrame({
    "customer_id": [f"c{i}" for i in range(N_CUSTOMERS)],
    "region": rng.choice(["north", "south", "east", "west"], N_CUSTOMERS),
    "segment": rng.choice(["budget", "mid", "premium"], N_CUSTOMERS),
})

products = pd.DataFrame({
    "product_id": [f"p{i}" for i in range(N_PRODUCTS)],
    "category": rng.choice(["electronics", "books", "clothing", "food"], N_PRODUCTS),
    "brand": rng.choice(["brandA", "brandB", "brandC"], N_PRODUCTS),
})

interactions = pd.DataFrame({
    "customer_id": rng.choice(customers["customer_id"].values, N_INTERACTIONS),
    "product_id": rng.choice(products["product_id"].values, N_INTERACTIONS),
    "weight": rng.uniform(0.1, 5.0, N_INTERACTIONS).round(2),
}).drop_duplicates(subset=["customer_id", "product_id"])

print(f"Customers: {len(customers)}, Products: {len(products)}, Interactions: {len(interactions)}")
interactions.head()

Customers: 50000, Products: 200, Interactions: 148872


,customer_id,product_id,weight
0,c12818,p185,2.58
1,c28278,p37,4.40
2,c25416,p193,3.12
3,c40438,p11,2.98
4,c17224,p30,2.00


## Fit model

In [12]:
model = WeightedSimilarity(
    w_product_metadata=0.25, w_product_interactions=0.25,
    w_customer_metadata=0.25, w_customer_interactions=0.25,
    metric="cosine",
    top_k=50,
)

%time model.fit(customers, products, interactions)
print(f"Score matrix R: shape={model._R.shape}, nnz={model._R.nnz}")

CPU times: total: 52.3 s
Wall time: 52.3 s
Score matrix R: shape=(50000, 200), nnz=10000000


In [15]:
model.recommend?

Signature:
model.recommend(
    customer_ids: 'list | pd.Index | None' = None,
    n: 'int' = 10,
) -> 'pd.DataFrame'
Docstring:
Return top-*n* product recommendations per customer.

Parameters
----------
customer_ids
    Subset of customers to score.  ``None`` means all customers
    seen during :meth:`fit`.
n
    Number of products to return per customer.

Returns
-------
DataFrame
    Columns ``customer_id``, ``product_id``, ``score``, ``rank``.
File:      c:\users\radek\desktop\recs\src\recs\models\weighted_similarity.py
Type:      method

In [13]:
model = WeightedSimilarity(
    w_product_metadata=0.25, w_product_interactions=0.25,
    w_customer_metadata=0.25, w_customer_interactions=0.25,
    metric="overlap",
    top_k=50,
)

%time model.fit(customers, products, interactions)
print(f"Score matrix R: shape={model._R.shape}, nnz={model._R.nnz}")

CPU times: total: 1min 9s
Wall time: 1min 9s
Score matrix R: shape=(50000, 200), nnz=10000000


## Get recommendations

In [14]:
recs = model.recommend( n=10)
recs

,customer_id,product_id,score,rank
0,c0,p176,12.730251,1
1,c0,p167,12.527911,2
2,c0,p190,12.199247,3
3,c0,p140,11.193904,4
4,c0,p162,11.156419,5
...,...,...,...,...
499995,c49999,p80,6.610097,6
499996,c49999,p164,6.439662,7
499997,c49999,p162,6.435903,8
499998,c49999,p175,6.434309,9


## Compare metrics

In [10]:
model_overlap = WeightedSimilarity(metric="overlap", top_k=50)
%time model_overlap.fit(customers, products, interactions)

recs_overlap = model_overlap.recommend(customer_ids=["c0"], n=10)
recs_cosine = model.recommend(customer_ids=["c0"], n=10)

comparison = recs_cosine[["product_id", "score"]].rename(columns={"score": "cosine_score"}).merge(
    recs_overlap[["product_id", "score"]].rename(columns={"score": "overlap_score"}),
    on="product_id", how="outer",
)
comparison

CPU times: total: 734 ms
Wall time: 735 ms


,product_id,cosine_score,overlap_score
0,p134,14.226693,NaN
1,p136,13.335255,9.479738
2,p151,13.704734,NaN
3,p161,NaN,9.805168
4,p168,NaN,9.446653
5,p170,15.194074,10.304576
6,p178,13.928872,10.109453
7,p185,13.491360,9.569700
8,p186,NaN,9.473764
9,p188,15.104571,10.207162


## Explore weight sensitivity

In [ ]:
configs = {
    "metadata_only": dict(
        w_product_metadata=1, w_product_interactions=0,
        w_customer_metadata=1, w_customer_interactions=0,
    ),
    "interaction_only": dict(
        w_product_metadata=0, w_product_interactions=1,
        w_customer_metadata=0, w_customer_interactions=1,
    ),
    "balanced": dict(
        w_product_metadata=0.25, w_product_interactions=0.25,
        w_customer_metadata=0.25, w_customer_interactions=0.25,
    ),
}

for name, weights in configs.items():
    m = WeightedSimilarity(**weights, metric="cosine", top_k=50)
    %time m.fit(customers, products, interactions)
    r = m.recommend(customer_ids=["c0"], n=5)
    print(f"\n--- {name} ---")
    print(r[["product_id", "score"]].to_string(index=False))